
## Đánh Giá Mô Hình B1 (YOLOv8n + P2) & Đối Sánh Toàn Diện Với Baseline B0

### 1. Mục tiêu
- **Đánh giá mô hình B1 trên Validation**: Đo lường chi tiết hiệu năng của mô hình $B_1$ ($YOLOv8n + P_2$) trên tập dữ liệu kiểm định `val.txt` (368 ảnh).
- **Đối sánh toàn diện B0 vs B1**: So sánh trực tiếp giữa Baseline $B_0$ và $B_1$ trên cùng một tập dữ liệu, cùng seed, cùng số epoch (100 epochs) và cùng quy trình tiền xử lý.
- **Trả lời câu hỏi nghiên cứu RQ2 (Chất lượng & Chi phí)**:
  - Việc bổ sung tầng phát hiện phân giải cao $P_2$ (stride 4, $160 \times 160$) có cải thiện Recall trên các khuyết tật nhỏ (`small` defects) và các lớp lỗi chí mạng (`open_circuit`, `short`) hay không?
  - Đổi lại mức cải thiện đó, chi phí tính toán (Parameters, GFLOPs, Latency) tăng thêm bao nhiêu?
- **Nguyên tắc bảo toàn dữ liệu**: Tuyệt đối **không dùng tập test (`test.txt`)** để tuning hay lựa chọn mô hình.

### 2. Import thư viện
Nạp các thư viện chuẩn phân tích dữ liệu, tính toán ma trận và mô hình thị giác: `pathlib`, `yaml`, `json`, `pandas`, `numpy`, `matplotlib`, `ultralytics.YOLO`.

In [1]:
from collections import defaultdict
from datetime import datetime, timezone
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
import yaml

def find_project_root() -> Path:
    candidate = Path.cwd().resolve()
    for parent in [candidate, *candidate.parents]:
        if (parent / "configs").is_dir() and (parent / "data").is_dir():
            return parent
    return candidate

ROOT = find_project_root()
os.chdir(ROOT)
print(f"Project root     : {ROOT}")
print(f"Working directory: {os.getcwd()}")
print(f"PyTorch version  : {torch.__version__}, Device: {'CUDA' if torch.cuda.is_available() else ('MPS' if torch.backends.mps.is_available() else 'CPU')}")

Project root     : /Users/Project/kltn-pcb
Working directory: /Users/Project/kltn-pcb
PyTorch version  : 2.11.0, Device: MPS


### 3. Load input
Nạp tất cả các tài nguyên cần thiết cho quá trình đánh giá và đối sánh:
- Checkpoint B1: `runs/B1/weights/best.pt`
- Metrics B0: `results/B0/metrics.json`
- B0 per-class recall: `results/B0/per_class_recall.csv`
- B0 size-bin recall: `results/B0/size_bin_recall.csv`
- Bảng chi phí phần cứng: `results/B1/architecture_cost.csv`
- Cấu hình dữ liệu: `data/data.yaml`
- Danh sách ảnh validation: `data/splits/val.txt`
- Bảng phân bố kích thước hộp: `results/size_distribution_boxes.csv`
- Cấu hình gốc: `configs/base.yaml`.

In [2]:
B1_BEST_PT = ROOT / "runs" / "B1_100ep" / "weights" / "best.pt"
if not B1_BEST_PT.is_file():
    B1_BEST_PT = ROOT / "runs" / "B1" / "weights" / "best.pt"
B0_METRICS_JSON = ROOT / "results" / "B0" / "metrics.json"
B0_PER_CLASS_CSV = ROOT / "results" / "B0" / "per_class_recall.csv"
B0_SIZE_BIN_CSV = ROOT / "results" / "B0" / "size_bin_recall.csv"
ARCH_COST_CSV = ROOT / "results" / "B1" / "architecture_cost.csv"
DATA_YAML_PATH = ROOT / "data" / "data.yaml"
VAL_TXT = ROOT / "data" / "splits" / "val.txt"
SIZE_DIST_CSV = ROOT / "results" / "size_distribution_boxes.csv"
CONFIG_PATH = ROOT / "configs" / "base.yaml"
RESULTS_B1_DIR = ROOT / "results" / "B1"
RESULTS_B1_DIR.mkdir(parents=True, exist_ok=True)

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    base_cfg = yaml.safe_load(f)

with open(DATA_YAML_PATH, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

with open(VAL_TXT, "r", encoding="utf-8") as f:
    val_img_paths = [line.strip() for line in f if line.strip()]

df_size_dist = pd.read_csv(SIZE_DIST_CSV)
df_arch_cost = pd.read_csv(ARCH_COST_CSV)

with open(B0_METRICS_JSON, "r", encoding="utf-8") as f:
    b0_metrics = json.load(f)

df_b0_per_class = pd.read_csv(B0_PER_CLASS_CSV)
df_b0_size_bin = pd.read_csv(B0_SIZE_BIN_CSV)

print("=== ĐÃ TẢI THÀNH CÔNG DỮ LIỆU ĐẦU VÀO ===")
print(f"Số lượng ảnh validation : {len(val_img_paths)} ảnh")
print(f"Số lớp khuyết tật       : {len(data_cfg.get('names', {}))} classes")
print(f"B0 mAP@0.5              : {b0_metrics.get('overall_metrics', {}).get('mAP_0.5', 0.0):.4f}")
print(f"B0 mAP@0.5:0.95         : {b0_metrics.get('overall_metrics', {}).get('mAP_0.5_0.95', 0.0):.4f}")

=== ĐÃ TẢI THÀNH CÔNG DỮ LIỆU ĐẦU VÀO ===
Số lượng ảnh validation : 368 ảnh
Số lớp khuyết tật       : 6 classes
B0 mAP@0.5              : 0.7471
B0 mAP@0.5:0.95         : 0.3018


### 4. Pre-flight check (Kiểm tra điều kiện tiên quyết)
Kiểm tra nghiêm ngặt tính toàn vẹn trước khi đánh giá:
1. Checkpoint B1 `best.pt` tồn tại và hợp lệ
2. Toàn bộ các file kết quả mốc của B0 tồn tại
3. Bảng `architecture_cost.csv` tồn tại
4. Tập `val.txt` tồn tại và đủ 368 ảnh
5. **Không dùng tập test (`test.txt`)** để đảm bảo bảo toàn dữ liệu.

In [3]:
assert B1_BEST_PT.is_file(), f"LỖI: Thiếu checkpoint B1: {B1_BEST_PT}"
assert B0_METRICS_JSON.is_file(), f"LỖI: Thiếu B0 metrics: {B0_METRICS_JSON}"
assert ARCH_COST_CSV.is_file(), f"LỖI: Thiếu architecture_cost.csv: {ARCH_COST_CSV}"
assert VAL_TXT.is_file(), f"LỖI: Thiếu file: {VAL_TXT}"
assert len(val_img_paths) == 368, f"Cảnh báo: val.txt phải có 368 ảnh, hiện có {len(val_img_paths)}"

TEST_TXT = ROOT / "data" / "splits" / "test.txt"
assert TEST_TXT.is_file(), f"Thiếu file {TEST_TXT}"
val_set = set(val_img_paths)
with open(TEST_TXT, "r", encoding="utf-8") as f:
    test_set = set(l.strip() for l in f if l.strip())
overlap = val_set & test_set
assert len(overlap) == 0, f"RÒ RỈ DỮ LIỆU: Có {len(overlap)} ảnh trùng giữa val và test!"

print("  [PASS] Checkpoint B1 best.pt tồn tại và sẵn sàng.")
print("  [PASS] Đầy đủ tài nguyên so sánh B0 và bảng chi phí phần cứng.")
print("  [PASS] Tập test được cô lập 100%, không bị xâm phạm.")

  [PASS] Checkpoint B1 best.pt tồn tại và sẵn sàng.
  [PASS] Đầy đủ tài nguyên so sánh B0 và bảng chi phí phần cứng.
  [PASS] Tập test được cô lập 100%, không bị xâm phạm.


### 5. Chạy Ultralytics Validation trên mô hình B1
- Khởi tạo mô hình từ checkpoint: `runs/B1/weights/best.pt`
- Thực thi `model.val(data="data/data.yaml", split="val", imgsz=640)`
- Trích xuất các chỉ số tổng quan: `mAP@0.5`, `mAP@0.5:0.95`, `Precision`, `Recall`.

In [4]:
model_b1 = YOLO(str(B1_BEST_PT))
print("=== BẮT ĐẦU CHẠY MODEL.VAL B1 TRÊN TẬP VALIDATION ===")
val_results_b1 = model_b1.val(
    data=str(DATA_YAML_PATH),
    split="val",
    imgsz=int(base_cfg.get("imgsz", 640)),
    verbose=True
)

b1_map50 = float(val_results_b1.box.map50)
b1_map = float(val_results_b1.box.map)
b1_precision = float(val_results_b1.box.mp)
b1_recall = float(val_results_b1.box.mr)

print("\n=== KẾT QUẢ TỔNG QUAN VALIDATION B1 (YOLOv8n + P2) ===")
print(f"Overall mAP@0.5     : {b1_map50:.4f}")
print(f"Overall mAP@0.5:0.95: {b1_map:.4f}")
print(f"Mean Precision      : {b1_precision:.4f}")
print(f"Mean Recall         : {b1_recall:.4f}")

=== BẮT ĐẦU CHẠY MODEL.VAL B1 TRÊN TẬP VALIDATION ===


Ultralytics 8.4.142 🚀 Python-3.13.5 torch-2.11.0 CPU (Apple M1 Pro)


YOLOv8n-p2 summary (fused): 90 layers, 2,921,832 parameters, 0 gradients, 12.2 GFLOPs


val: Fast image access ✅ (ping: 0.3±0.4 ms, read: 1688.6±889.9 MB/s, size: 1633.0 KB)



val: Scanning data/raw/PKU-Market-PCB(Data enhanced version)/train/labels.cache... 368 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 368/368 30.3Mit/s 0.0s

...

                  spur         60        300      0.556      0.537      0.511      0.219


       spurious_copper         64        328      0.927      0.637       0.78      0.298


Speed: 0.6ms preprocess, 273.4ms inference, 0.0ms loss, 1.1ms postprocess per image


Results saved to /Users/Project/kltn-pcb/runs/detect/val-5



=== KẾT QUẢ TỔNG QUAN VALIDATION B1 (YOLOv8n + P2) ===
Overall mAP@0.5     : 0.7034
Overall mAP@0.5:0.95: 0.2832
Mean Precision      : 0.7128
Mean Recall         : 0.6797


### 6. Chạy dự đoán chi tiết trên Validation Set để phục vụ phân tích sâu
- Chạy `model.predict()` trên toàn bộ 368 ảnh trong `val.txt` với ngưỡng tiêu chuẩn (`conf=0.25`, `iou=0.7`, `imgsz=640`).
- Thu thập chi tiết từng bounding box dự đoán (tọa độ pixel `xyxy`, confidence score, `class_id`, `class_name`).
- Lưu toàn bộ dự đoán vào `results/B1/val_predictions.csv`.

In [5]:
val_full_paths = [str(ROOT / p) for p in val_img_paths]
print(f"Đang thực hiện inference B1 trên {len(val_full_paths)} ảnh validation...")

preds_batch = model_b1.predict(
    source=val_full_paths,
    conf=0.25,
    iou=0.7,
    imgsz=int(base_cfg.get("imgsz", 640)),
    save=False,
    verbose=False,
)

pred_records = []
for img_rel, res in zip(val_img_paths, preds_batch):
    boxes = res.boxes
    if boxes is not None and len(boxes) > 0:
        for b in boxes:
            xyxy = b.xyxy[0].cpu().numpy()
            conf = float(b.conf[0].cpu().numpy())
            cls_id = int(b.cls[0].cpu().numpy())
            pred_records.append({
                "image_path": img_rel,
                "image_name": Path(img_rel).name,
                "class_id": cls_id,
                "class_name": data_cfg["names"][cls_id],
                "confidence": conf,
                "xmin": float(xyxy[0]),
                "ymin": float(xyxy[1]),
                "xmax": float(xyxy[2]),
                "ymax": float(xyxy[3]),
            })

df_b1_preds = pd.DataFrame(pred_records)
VAL_PREDS_CSV = RESULTS_B1_DIR / "val_predictions.csv"
df_b1_preds.to_csv(VAL_PREDS_CSV, index=False)
print(f"Đã lưu {len(df_b1_preds)} predictions vào {VAL_PREDS_CSV.relative_to(ROOT)}")

Đang thực hiện inference B1 trên 368 ảnh validation...


Đã lưu 1973 predictions vào results/B1/val_predictions.csv


### 7. Tính toán và lưu trữ các bộ chỉ số đo lường chi tiết của B1
1. **Per-class Recall**: Chi tiết Precision, Recall, F1, mAP50 cho 6 lớp khuyết tật (`results/B1/per_class_recall.csv`).
2. **Size-bin Recall (RQ2)**: Đối chiếu IoU $\ge 0.5$ với Ground-Truth để tính Recall riêng cho `small`, `medium`, `large` (`results/B1/size_bin_recall.csv`).
3. **Critical Defect FN**: Đo lường False Negative Rate trên hai lớp khuyết tật chí mạng `open_circuit` và `short` (`results/B1/critical_fn.csv`).
4. **Board-level Metrics**: Tính tỷ lệ lọt bo lỗi (False Accept Rate / Escape Rate) (`results/B1/board_metrics.csv`).
5. **Biểu đồ trực quan hóa**: Biểu đồ cột so sánh Recall theo size bin (`results/B1/recall_by_size_bin.png`).
6. **Tổng hợp JSON**: Đóng gói toàn bộ metrics vào `results/B1/metrics.json`.

In [6]:
names_dict = data_cfg["names"]
critical_classes = ["open_circuit", "short"]
critical_class_ids = [k for k, v in names_dict.items() if v in critical_classes]

# 1. Per-class metrics
p_b1 = val_results_b1.box.p
r_b1 = val_results_b1.box.r
f1_b1 = val_results_b1.box.f1
ap50_b1 = val_results_b1.box.ap50
ap_b1 = val_results_b1.box.ap

per_class_records = []
for cid in sorted(names_dict.keys()):
    cname = names_dict[cid]
    p_val = float(p_b1[cid]) if cid < len(p_b1) else 0.0
    r_val = float(r_b1[cid]) if cid < len(r_b1) else 0.0
    f1_val = float(f1_b1[cid]) if cid < len(f1_b1) else 0.0
    ap50_val = float(ap50_b1[cid]) if cid < len(ap50_b1) else 0.0
    ap_val = float(ap_b1[cid]) if cid < len(ap_b1) else 0.0
    per_class_records.append({
        "class_id": cid,
        "class_name": cname,
        "precision": round(p_val, 4),
        "recall": round(r_val, 4),
        "f1": round(f1_val, 4),
        "mAP50": round(ap50_val, 4),
        "mAP50_95": round(ap_val, 4),
    })

df_b1_per_class = pd.DataFrame(per_class_records)
B1_PER_CLASS_CSV = RESULTS_B1_DIR / "per_class_recall.csv"
df_b1_per_class.to_csv(B1_PER_CLASS_CSV, index=False)
print(f"Đã lưu per-class recall vào: {B1_PER_CLASS_CSV.relative_to(ROOT)}")

# 2. Size-bin recall matching
val_basenames_set = set(Path(p).name for p in val_img_paths)
df_size_dist["image_name"] = df_size_dist["image_path"].apply(lambda p: Path(p).name)
df_val_gt = df_size_dist[df_size_dist["image_name"].isin(val_basenames_set)].copy()

def compute_iou(box1, box2):
    x1, y1 = max(box1[0], box2[0]), max(box1[1], box2[1])
    x2, y2 = min(box1[2], box2[2]), min(box1[3], box2[3])
    inter_w, inter_h = max(0.0, x2 - x1), max(0.0, y2 - y1)
    inter_area = inter_w * inter_h
    area1 = max(0.0, (box1[2] - box1[0]) * (box1[3] - box1[1]))
    area2 = max(0.0, (box2[2] - box2[0]) * (box2[3] - box2[1]))
    union_area = area1 + area2 - inter_area
    return inter_area / union_area if union_area > 0 else 0.0

preds_by_img = defaultdict(list)
for _, row in df_b1_preds.iterrows():
    preds_by_img[row["image_name"]].append(row)

gt_matched = []
for _, gt_row in df_val_gt.iterrows():
    img_name = gt_row["image_name"]
    gt_cid = int(gt_row["class_id"])
    orig_w, orig_h = float(gt_row["original_width_px"]), float(gt_row["original_height_px"])
    lbl_file = ROOT / "data/processed/labels" / Path(gt_row["label_path"])
    if not lbl_file.is_file():
        lbl_file = ROOT / "data/raw/PKU-Market-PCB(Data enhanced version)" / Path(gt_row["label_path"])
    
    line_idx = int(gt_row["label_line"]) - 1
    with open(lbl_file, "r", encoding="utf-8") as fp:
        lines = [l.strip() for l in fp if l.strip()]
    parts = list(map(float, lines[line_idx].split()))
    xc, yc, w_norm, h_norm = parts[1], parts[2], parts[3], parts[4]
    gt_box = [(xc - w_norm/2)*orig_w, (yc - h_norm/2)*orig_h, (xc + w_norm/2)*orig_w, (yc + h_norm/2)*orig_h]
    
    best_iou = 0.0
    for p in preds_by_img.get(img_name, []):
        if int(p["class_id"]) == gt_cid:
            p_box = [p["xmin"], p["ymin"], p["xmax"], p["ymax"]]
            iou = compute_iou(gt_box, p_box)
            if iou > best_iou:
                best_iou = iou
    gt_matched.append(best_iou >= 0.5)

df_val_gt["detected"] = gt_matched

size_bin_records = []
for sbin in ["small", "medium", "large"]:
    sub = df_val_gt[df_val_gt["size_bin"] == sbin]
    n_total = len(sub)
    n_det = sub["detected"].sum()
    rec = (n_det / n_total) if n_total > 0 else 0.0
    size_bin_records.append({
        "size_bin": sbin,
        "total_boxes": n_total,
        "detected_boxes": int(n_det),
        "missed_boxes": int(n_total - n_det),
        "recall": round(float(rec), 4),
        "fn_rate": round(float(1.0 - rec if n_total > 0 else 0.0), 4),
    })

df_b1_size_bin = pd.DataFrame(size_bin_records)
B1_SIZE_BIN_CSV = RESULTS_B1_DIR / "size_bin_recall.csv"
df_b1_size_bin.to_csv(B1_SIZE_BIN_CSV, index=False)
print(f"Đã lưu size-bin recall vào: {B1_SIZE_BIN_CSV.relative_to(ROOT)}")

# 3. Critical FN
crit_records = []
for cid in critical_class_ids:
    cname = names_dict[cid]
    sub = df_val_gt[df_val_gt["class_id"] == cid]
    n_tot = len(sub)
    n_d = sub["detected"].sum()
    crit_records.append({
        "class_id": cid,
        "class_name": cname,
        "total_gt": n_tot,
        "detected_tp": int(n_d),
        "missed_fn": int(n_tot - n_d),
        "fn_rate": round(float((n_tot - n_d) / n_tot) if n_tot > 0 else 0.0, 4),
        "recall": round(float(n_d / n_tot) if n_tot > 0 else 0.0, 4),
    })
sub_all_crit = df_val_gt[df_val_gt["class_id"].isin(critical_class_ids)]
tot_c = len(sub_all_crit)
det_c = sub_all_crit["detected"].sum()
crit_records.append({
    "class_id": -1,
    "class_name": "ALL_CRITICAL",
    "total_gt": tot_c,
    "detected_tp": int(det_c),
    "missed_fn": int(tot_c - det_c),
    "fn_rate": round(float((tot_c - det_c) / tot_c) if tot_c > 0 else 0.0, 4),
    "recall": round(float(det_c / tot_c) if tot_c > 0 else 0.0, 4),
})
df_b1_crit = pd.DataFrame(crit_records)
B1_CRIT_CSV = RESULTS_B1_DIR / "critical_fn.csv"
df_b1_crit.to_csv(B1_CRIT_CSV, index=False)
print(f"Đã lưu critical FN vào: {B1_CRIT_CSV.relative_to(ROOT)}")

# 4. Board-level metrics
tot_boards = len(val_img_paths)
det_boards = len(set(df_b1_preds["image_path"].unique()))
esc_boards = tot_boards - det_boards
df_b1_board = pd.DataFrame([{
    "total_boards": tot_boards,
    "ground_truth_defective": tot_boards,
    "ground_truth_pass": 0,
    "predicted_defective": det_boards,
    "predicted_pass_escaped": esc_boards,
    "false_accept_rate": round(float(esc_boards / tot_boards), 4),
    "false_reject_rate": "N/A (no defect-free images)",
    "board_detection_accuracy": round(float(det_boards / tot_boards), 4),
}])
B1_BOARD_CSV = RESULTS_B1_DIR / "board_metrics.csv"
df_b1_board.to_csv(B1_BOARD_CSV, index=False)
print(f"Đã lưu board metrics vào: {B1_BOARD_CSV.relative_to(ROOT)}")

# 5. Metrics JSON
b1_metrics_dict = {
    "evaluation_target": "B1_YOLOv8n_P2",
    "split_evaluated": "validation",
    "num_validation_images": tot_boards,
    "evaluation_timestamp": datetime.now(timezone.utc).isoformat(),
    "overall_metrics": {
        "mAP_0.5": b1_map50,
        "mAP_0.5_0.95": b1_map,
        "precision_mean": b1_precision,
        "recall_mean": b1_recall
    },
    "precision_per_class": {r["class_name"]: r["precision"] for _, r in df_b1_per_class.iterrows()},
    "recall_per_class": {r["class_name"]: r["recall"] for _, r in df_b1_per_class.iterrows()},
    "f1_per_class": {r["class_name"]: r["f1"] for _, r in df_b1_per_class.iterrows()},
    "size_bin_recall": {r["size_bin"]: r["recall"] for _, r in df_b1_size_bin.iterrows()},
    "critical_defects": {
        "critical_classes": critical_classes,
        "critical_defect_fn_rate": float(df_b1_crit[df_b1_crit["class_name"]=="ALL_CRITICAL"]["fn_rate"].iloc[0]),
        "critical_defect_recall": float(df_b1_crit[df_b1_crit["class_name"]=="ALL_CRITICAL"]["recall"].iloc[0])
    },
    "board_level_metrics": {
        "false_accept_rate": float(df_b1_board["false_accept_rate"].iloc[0]),
        "false_reject_rate": None,
        "limitation_note": "Dataset PKU-Market-PCB validation split contains 0 defect-free boards; False Reject Rate cannot be evaluated."
    }
}
B1_METRICS_JSON = RESULTS_B1_DIR / "metrics.json"
with open(B1_METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(b1_metrics_dict, f, indent=2)
print(f"Đã lưu metrics JSON vào: {B1_METRICS_JSON.relative_to(ROOT)}")

# 6. Biểu đồ trực quan hóa
fig, ax = plt.subplots(figsize=(8, 5))
bins = ["small", "medium", "large"]
x = np.arange(len(bins))
width = 0.35
r_b0_vals = [float(df_b0_size_bin[df_b0_size_bin["size_bin"]==b]["recall"].iloc[0]) if len(df_b0_size_bin[df_b0_size_bin["size_bin"]==b])>0 else 0.0 for b in bins]
r_b1_vals = [float(df_b1_size_bin[df_b1_size_bin["size_bin"]==b]["recall"].iloc[0]) for b in bins]

rects1 = ax.bar(x - width/2, r_b0_vals, width, label="B0 (YOLOv8n)", color="#3498db")
rects2 = ax.bar(x + width/2, r_b1_vals, width, label="B1 (YOLOv8n + P2)", color="#2ecc71")
ax.set_ylabel("Recall")
ax.set_title("So Sánh Recall Theo Size Bin Giữa B0 và B1 (RQ2)")
ax.set_xticks(x)
ax.set_xticklabels(bins)
ax.set_ylim(0, 1.0)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)

for bar in rects1 + rects2:
    h = bar.get_height()
    ax.annotate(f"{h:.1%}", xy=(bar.get_x() + bar.get_width() / 2, h), xytext=(0, 3),
                textcoords="offset points", ha="center", va="bottom", fontsize=9)

chart_path = RESULTS_B1_DIR / "recall_by_size_bin.png"
plt.tight_layout()
plt.savefig(chart_path, dpi=300)
plt.close()
print(f"Đã lưu biểu đồ vào: {chart_path.relative_to(ROOT)}")

Đã lưu per-class recall vào: results/B1/per_class_recall.csv


Đã lưu size-bin recall vào: results/B1/size_bin_recall.csv
Đã lưu critical FN vào: results/B1/critical_fn.csv
Đã lưu board metrics vào: results/B1/board_metrics.csv
Đã lưu metrics JSON vào: results/B1/metrics.json


Đã lưu biểu đồ vào: results/B1/recall_by_size_bin.png


### 8. Lập bảng đối sánh chuẩn B0 vs B1 (3 dòng: B0, B1_P2, Delta)
Bảng tổng hợp đối sánh các chỉ số cốt lõi:
- Cột: `config`, `mAP@0.5`, `mAP@0.5:0.95`, `recall_small`, `recall_medium`, `recall_large`, `recall_open_circuit`, `recall_short`, `params`, `GFLOPs`.
- Dòng: `B0`, `B1_P2`, `Delta` (chênh lệch tuyệt đối và phần trăm).
- Xuất kết quả ra `results/B1/b0_b1_comparison.csv` và `results/B1/b0_b1_comparison.md`.

In [7]:
# Lấy thông số từ architecture_cost.csv
b0_cost_row = df_arch_cost[df_arch_cost["config"].str.contains("B0")].iloc[0]
b1_cost_row = df_arch_cost[df_arch_cost["config"].str.contains("B1")].iloc[0]

b0_params = int(b0_cost_row["params"])
b0_gflops = float(b0_cost_row["gflops"])
b1_params = int(b1_cost_row["params"])
b1_gflops = float(b1_cost_row["gflops"])

# Lấy recall theo lớp từ per_class
b0_rec_open = float(df_b0_per_class[df_b0_per_class["class_name"]=="open_circuit"]["recall"].iloc[0])
b0_rec_short = float(df_b0_per_class[df_b0_per_class["class_name"]=="short"]["recall"].iloc[0])
b1_rec_open = float(df_b1_per_class[df_b1_per_class["class_name"]=="open_circuit"]["recall"].iloc[0])
b1_rec_short = float(df_b1_per_class[df_b1_per_class["class_name"]=="short"]["recall"].iloc[0])

# Lấy recall size-bin
b0_r_s = float(df_b0_size_bin[df_b0_size_bin["size_bin"]=="small"]["recall"].iloc[0])
b0_r_m = float(df_b0_size_bin[df_b0_size_bin["size_bin"]=="medium"]["recall"].iloc[0])
b0_r_l = float(df_b0_size_bin[df_b0_size_bin["size_bin"]=="large"]["recall"].iloc[0]) if len(df_b0_size_bin[df_b0_size_bin["size_bin"]=="large"])>0 else 0.0

b1_r_s = float(df_b1_size_bin[df_b1_size_bin["size_bin"]=="small"]["recall"].iloc[0])
b1_r_m = float(df_b1_size_bin[df_b1_size_bin["size_bin"]=="medium"]["recall"].iloc[0])
b1_r_l = float(df_b1_size_bin[df_b1_size_bin["size_bin"]=="large"]["recall"].iloc[0]) if len(df_b1_size_bin[df_b1_size_bin["size_bin"]=="large"])>0 else 0.0

b0_map50 = float(b0_metrics["overall_metrics"]["mAP_0.5"])
b0_map = float(b0_metrics["overall_metrics"]["mAP_0.5_0.95"])

# Dòng Delta
delta_map50 = b1_map50 - b0_map50
delta_map = b1_map - b0_map
delta_r_s = b1_r_s - b0_r_s
delta_r_m = b1_r_m - b0_r_m
delta_r_l = b1_r_l - b0_r_l
delta_rec_open = b1_rec_open - b0_rec_open
delta_rec_short = b1_rec_short - b0_rec_short
delta_params = b1_params - b0_params
delta_gflops = b1_gflops - b0_gflops

comparison_records = [
    {
        "config": "B0",
        "mAP@0.5": round(b0_map50, 4),
        "mAP@0.5:0.95": round(b0_map, 4),
        "recall_small": round(b0_r_s, 4),
        "recall_medium": round(b0_r_m, 4),
        "recall_large": round(b0_r_l, 4),
        "recall_open_circuit": round(b0_rec_open, 4),
        "recall_short": round(b0_rec_short, 4),
        "params": b0_params,
        "GFLOPs": round(b0_gflops, 2)
    },
    {
        "config": "B1_P2",
        "mAP@0.5": round(b1_map50, 4),
        "mAP@0.5:0.95": round(b1_map, 4),
        "recall_small": round(b1_r_s, 4),
        "recall_medium": round(b1_r_m, 4),
        "recall_large": round(b1_r_l, 4),
        "recall_open_circuit": round(b1_rec_open, 4),
        "recall_short": round(b1_rec_short, 4),
        "params": b1_params,
        "GFLOPs": round(b1_gflops, 2)
    },
    {
        "config": "Delta",
        "mAP@0.5": f"{delta_map50:+.4f}",
        "mAP@0.5:0.95": f"{delta_map:+.4f}",
        "recall_small": f"{delta_r_s:+.4f}",
        "recall_medium": f"{delta_r_m:+.4f}",
        "recall_large": f"{delta_r_l:+.4f}",
        "recall_open_circuit": f"{delta_rec_open:+.4f}",
        "recall_short": f"{delta_rec_short:+.4f}",
        "params": f"{delta_params:+d} ({delta_params/b0_params:+.1%})",
        "GFLOPs": f"{delta_gflops:+.2f} ({delta_gflops/b0_gflops:+.1%})"
    }
]

df_comp = pd.DataFrame(comparison_records)
COMP_CSV = RESULTS_B1_DIR / "b0_b1_comparison.csv"
COMP_MD = RESULTS_B1_DIR / "b0_b1_comparison.md"

df_comp.to_csv(COMP_CSV, index=False)
md_table = df_comp.to_markdown(index=False)
with open(COMP_MD, "w", encoding="utf-8") as f:
    f.write("# Bảng Đối Sánh Thực Nghiệm B0 vs B1 (YOLOv8n vs YOLOv8n-P2)\n\n" + md_table + "\n")

print("=== BẢNG ĐỐI SÁNH HIỆU NĂNG B0 VS B1 (RQ2) ===")
print(df_comp.to_string(index=False))
print(f"\nĐã xuất kết quả sang:\n  - {COMP_CSV.relative_to(ROOT)}\n  - {COMP_MD.relative_to(ROOT)}")

=== BẢNG ĐỐI SÁNH HIỆU NĂNG B0 VS B1 (RQ2) ===
config mAP@0.5 mAP@0.5:0.95 recall_small recall_medium recall_large recall_open_circuit recall_short         params         GFLOPs
    B0  0.7471       0.3018       0.7465           0.7          0.0              0.6883       0.7007        3012018            8.2
 B1_P2  0.7034       0.2832       0.6535        0.5207          0.0              0.6464       0.7059        2927352          12.36
 Delta -0.0437      -0.0186      -0.0930       -0.1793      +0.0000             -0.0419      +0.0052 -84666 (-2.8%) +4.16 (+50.7%)

Đã xuất kết quả sang:
  - results/B1/b0_b1_comparison.csv
  - results/B1/b0_b1_comparison.md


### 9. Báo cáo đánh giá khoa học trung thực (`results/B1/eval_summary.md`)
Phân tích trung thực kết quả thực nghiệm:
- Đánh giá khách quan mức tăng/giảm Recall trên các khuyết tật nhỏ (`small`).
- Đối chiếu với chi phí tính toán tăng thêm (GFLOPs tăng +50.8%).
- Nếu $B_1$ tốt hơn: Định lượng chính xác tỷ lệ cải thiện và nguyên nhân kiến trúc (nhánh $P_2$ bảo toàn spatial details $160 \times 160$).
- Nếu $B_1$ không cải thiện hoặc kém hơn: Ghi nhận trung thực, không suy diễn kết luận sai lệch, đưa ra các giả thuyết khoa học (ví dụ: cần thêm epoch để hội tụ do receptive field nhỏ, hay cần anchor/DFL tuning).

In [8]:
summary_lines = []
summary_lines.append("# BÁO CÁO ĐÁNH GIÁ THỰC NGHIỆM B1 (YOLOv8n + P2) VÀ ĐỐI SÁNH VỚI B0")
summary_lines.append(f"\n**Thời gian thẩm định**: {datetime.now(timezone.utc).isoformat()}")
summary_lines.append("**Tập dữ liệu đánh giá**: `val.txt` (368 ảnh, 1,860 GT boxes - hoàn toàn độc lập với tập test)\n")
summary_lines.append("## 1. Tóm tắt kết quả cốt lõi\n")
summary_lines.append(md_table + "\n")
summary_lines.append("## 2. Phân tích trả lời câu hỏi nghiên cứu RQ2\n")

if delta_r_s > 0:
    summary_lines.append(f"- **Về năng lực phát hiện vật thể nhỏ (Recall Small)**: Mô hình B1 cải thiện Recall trên nhóm lỗi nhỏ thêm **{delta_r_s:+.2%}** (từ {b0_r_s:.2%} lên {b1_r_s:.2%}). Điều này chứng minh giả thuyết tầng P2 ($160 \times 160$, stride 4) giúp mô hình giữ lại các đặc trưng biên dạng vi mô của khuyết tật PCB tốt hơn tầng P3 tiêu chuẩn.")
elif delta_r_s == 0:
    summary_lines.append(f"- **Về năng lực phát hiện vật thể nhỏ (Recall Small)**: Mô hình B1 đạt mức Recall tương đương B0 ({b1_r_s:.2%}).")
else:
    summary_lines.append(f"- **Về năng lực phát hiện vật thể nhỏ (Recall Small)**: Mô hình B1 có Recall trên nhóm lỗi nhỏ chênh lệch **{delta_r_s:+.2%}** so với B0 ({b1_r_s:.2%} vs {b0_r_s:.2%}).")
    summary_lines.append("  - *Giả thuyết khoa học*: Tầng P2 trích xuất đặc trưng ở độ phân giải lớn ($160 \times 160$) nhưng trường thụ cảm (receptive field) hẹp, đòi hỏi mô hình cần nhiều bước gradient hơn (epoch budget lớn hơn) để hội tụ tối ưu, hoặc cần cơ chế chú ý định hướng.")

summary_lines.append(f"\n- **Về các khuyết tật chí mạng (Critical Defects)**:")
summary_lines.append(f"  - `open_circuit`: B0 đạt {b0_rec_open:.2%}, B1 đạt {b1_rec_open:.2%} (Chênh lệch: {delta_rec_open:+.2%}).")
summary_lines.append(f"  - `short`       : B0 đạt {b0_rec_short:.2%}, B1 đạt {b1_rec_short:.2%} (Chênh lệch: {delta_rec_short:+.2%}).")

summary_lines.append(f"\n- **Về chỉ số tổng quan mAP@0.5**: B0 đạt {b0_map50:.4f}, B1 đạt {b1_map50:.4f} (Chênh lệch: {delta_map50:+.4f}).")
summary_lines.append(f"- **Về chi phí tính toán đánh đổi**:")
summary_lines.append(f"  - Số tham số (Parameters): B0 có {b0_params:,} params, B1 có {b1_params:,} params ({delta_params/b0_params:+.2%}).")
summary_lines.append(f"  - Độ phức tạp tính toán (GFLOPs): B0 tiêu tốn {b0_gflops:.2f} GFLOPs, B1 tiêu tốn {b1_gflops:.2f} GFLOPs ({delta_gflops/b0_gflops:+.2%}).")
summary_lines.append("\n## 3. Kết luận và định hướng tiếp theo\n")
summary_lines.append("- Thí nghiệm B1 đã hoàn tất việc cô lập biến: chỉ thay đổi duy nhất kiến trúc mạng (bổ sung nhánh P2), không can thiệp split, không dùng test set để tuning.")
summary_lines.append("- Kết quả đối sánh này là căn cứ thực nghiệm vững chắc để đánh giá hiệu quả của việc can thiệp kiến trúc FPN phân giải cao trước khi bước vào các giai đoạn tiếp theo.")

summary_content = "\n".join(summary_lines) + "\n"
SUMMARY_MD = RESULTS_B1_DIR / "eval_summary.md"
with open(SUMMARY_MD, "w", encoding="utf-8") as f:
    f.write(summary_content)

print(f"Đã lưu báo cáo đánh giá khoa học vào: {SUMMARY_MD.relative_to(ROOT)}")
print("\n" + summary_content)

Đã lưu báo cáo đánh giá khoa học vào: results/B1/eval_summary.md

# BÁO CÁO ĐÁNH GIÁ THỰC NGHIỆM B1 (YOLOv8n + P2) VÀ ĐỐI SÁNH VỚI B0

**Thời gian thẩm định**: 2026-09-25T05:23:58.374892+00:00
**Tập dữ liệu đánh giá**: `val.txt` (368 ảnh, 1,860 GT boxes - hoàn toàn độc lập với tập test)

## 1. Tóm tắt kết quả cốt lõi

| config   |   mAP@0.5 |   mAP@0.5:0.95 |   recall_small |   recall_medium |   recall_large |   recall_open_circuit |   recall_short | params         | GFLOPs         |
|:---------|----------:|---------------:|---------------:|----------------:|---------------:|----------------------:|---------------:|:---------------|:---------------|
| B0       |    0.7471 |         0.3018 |         0.7465 |          0.7    |              0 |                0.6883 |         0.7007 | 3012018        | 8.2            |
| B1_P2    |    0.7034 |         0.2832 |         0.6535 |          0.5207 |              0 |                0.6464 |         0.7059 | 2927352        | 12.36          |
| De

### 10. Ghi nhận nhật ký quyết định kỹ thuật (`docs/decision_log.md`)
Append kết quả thẩm định và bảng đối sánh B0 vs B1 vào nhật ký quyết định của đề tài.

In [9]:
decision_log_path = ROOT / "docs" / "decision_log.md"
now_date = datetime.now().strftime("%Y-%m-%d")

log_entry = f"""
## {now_date}: Đánh Giá Mô Hình B1 (YOLOv8n + P2) và Đối Sánh Hiệu Năng Với B0 (Tuần 8)

- Quyết định: Hoàn tất đánh giá toàn diện mô hình B1 trên validation set (368 ảnh) và xuất bảng đối sánh hiệu năng với Baseline B0.
- Bảng Đối Sánh B0 vs B1 (Validation):
{md_table}
- Nhận định sơ bộ (Validation):
  - mAP@0.5: B0 = {b0_map50:.4f}, B1 = {b1_map50:.4f} (Chênh lệch: {delta_map50:+.4f}).
  - Recall Small: B0 = {b0_r_s:.2%}, B1 = {b1_r_s:.2%} (Chênh lệch: {delta_r_s:+.2%}).
  - Đánh đổi chi phí: GFLOPs tăng từ {b0_gflops:.2f} lên {b1_gflops:.2f} ({delta_gflops/b0_gflops:+.1%}).
- Ràng buộc kiểm chứng: Toàn bộ quá trình đánh giá chỉ thực hiện trên tập `val.txt`; tuyệt đối **không sử dụng tập test (`data/splits/test.txt`) để tuning** hay lựa chọn mô hình.
"""

with open(decision_log_path, "a", encoding="utf-8") as f:
    f.write(log_entry)

print(f"Đã append thành công kết quả đối sánh B1 vs B0 vào {decision_log_path.relative_to(ROOT)}")

Đã append thành công kết quả đối sánh B1 vs B0 vào docs/decision_log.md
